In [ ]:
#Getting the needed libraries and dependencies
!pip uninstall -y transformers tokenizers IndicTransToolkit

!pip install transformers==4.39.0 tokenizers==0.15.0
!pip install git+https://github.com/VarunGumma/IndicTransToolkit.git
!pip install sacremoses sentencepiece datasets accelerate

import os
os.kill(os.getpid(), 9)

In [ ]:
#Verification
try:
    from IndicTransToolkit import IndicProcessor
    import torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    print("Success! IndicProcessor and Transformers are ready.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
#Mount Google Drive
drive.mount('/content/drive')

In [ ]:
#Tier 1
import torch
import pandas as pd
import os
from google.colab import drive
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor
from tqdm import tqdm

SAVE_DIR = "YOUR_SAVE_PATH" #Output file is stored here
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ai4bharat/indictrans2-indic-en-1B" #Translation Model
ip = IndicProcessor(inference=True)

print(f"Loading Model on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(DEVICE)

dataset = load_dataset("ai4bharat/MILU", data_dir="Your_Chosen_Language", split="Test_Or_Validation_Split") #MILU Dataset
df_full = pd.DataFrame(dataset)
questions_list = df_full['question'].tolist() #Column containing the questions in the chosen language

all_translations = []
batch_size = 32

print(f"Translating {len(questions_list)} questions...")

for i in tqdm(range(0, len(questions_list), batch_size)):
    batch = questions_list[i : i + batch_size]

    preprocessed_batch = ip.preprocess_batch(
        batch,
        src_lang="Your_Chosen_Language_Code",  #Refer Github ReadMe
        tgt_lang="eng_Latn"
    )

    inputs = tokenizer(
        preprocessed_batch,
        truncation=True,
        padding="longest",
        max_length=512,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            max_length=512,
            num_beams=5,
            use_cache=True
        )

    decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    post_processed = ip.postprocess_batch(decoded, lang="eng_Latn") #Type Translate_To_Language_Code here too
    all_translations.extend(post_processed)

df_full["question_en"] = all_translations #Column where translated questions are stored

csv_path = os.path.join(SAVE_DIR, "TYPE_FILE_NAME.csv")
df_full.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f"Success! File saved to your Google Drive at: {csv_path}")